In [ ]:
#https://maykosilva.com/blog/projeto-pratico-de-regressao-com-python-e-scikit-learn/
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures


In [ ]:
df = pd.read_csv('polynomial_ridge_dataset.csv')
print(df.head())

In [ ]:
valores_faltantes = df.isnull().sum()
print(valores_faltantes)

numeric_columns = df.select_dtypes(include=[np.number])

num_subplot = len(numeric_columns.columns)
num_rows = (num_subplot + 1) // 2
num_cols = 2

figs, axes = plt.subplots(num_rows, num_cols, figsize=(12, num_rows*4))
figs.suptitle('Histogramas das colunas numéricas', fontsize=16)
for i, coluna in enumerate(numeric_columns.columns):
    row = i // num_cols
    col = i % num_cols
    ax = axes[row, col]
    ax.hist(numeric_columns[coluna], bins=20)
    ax.set_title(coluna)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("Estasticas: ")
print(df.describe())


In [ ]:
#Verificação se tem numeros invalidos

df['target'] = pd.to_numeric(df['target'], errors='coerce')
total_nan = df['target'].isnull().sum()
print("Total de NaN em Target Column: ", total_nan)

df['x1'] = pd.to_numeric(df['x1'], errors='coerce')
total_nan = df['x1'].isnull().sum()
print("Total de NaN em x1 Column: ", total_nan)

df['x2'] = pd.to_numeric(df['x2'], errors='coerce')
total_nan = df['x2'].isnull().sum()
print("Total de NaN em x2 Column: ", total_nan)


In [ ]:
x = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=0)

In [ ]:
numeric_columns = ['x1', 'x2']

X_train_nums = X_train[numeric_columns].apply(pd.to_numeric, errors='coerce')
matriz = X_train_nums.corr()
print(matriz)

          x1        x2
x1  1.000000 -0.019149
x2 -0.019149  1.000000


In [ ]:
colums_to_plot = ['x1', 'x2']

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(8, 8))
fig.suptitle("Matriz correlação")
fig.tight_layout(pad=0.5)

for i, column in enumerate(colums_to_plot):
  row = i // 2
  col = i % 2
  ax = axes[row, col]
  ax.scatter(X_train_nums[column], y_train)
  ax.set_title(f"{column} vs. target")
  ax.set_xlabel(column)
  ax.set_ylabel("target")
plt.show()


In [ ]:
standard_scaler = StandardScaler()
X_train_scaled = standard_scaler.fit_transform(X_train[numeric_columns])
poly_features_test_transformer = PolynomialFeatures(degree=2)
X_train_poly = poly_features_test_transformer.fit_transform(X_train_scaled)

model = Ridge(alpha=1.0)

cvs = cross_val_score(model, X_train_poly, y_train, cv=5)
valores = np.abs(cvs)

print("Cross Val Score valores: ", valores)
print("Média dos resultados: ", valores.mean())

model.fit(X_train_poly, y_train)

print("Modelo coeficientes: ", model.coef_)
print("Modelos intercepto: ", model.intercept_)


In [ ]:
train_predict = model.predict(X_train_poly)

mse = mean_squared_error(y_train, train_predict)
r2 = r2_score(y_train, train_predict)

print("\nPrevisões no conjunto de treino: ")
print("Raiz de MSE: ", np.sqrt(mse))
print("R2: ", r2)

#Previsão no conjunto de teste
X_test_scaled = standard_scaler.transform(X_test[numeric_columns])
X_test_poly = poly_features_test_transformer.transform(X_test_scaled)

test_predict = model.predict(X_test_poly)

mse_teste = mean_squared_error(y_test, test_predict)
r2_teste = r2_score(y_test, test_predict)

print("\nPrevisões no conjunto de teste: ")
print("Raiz de MSE: ", np.sqrt(mse_teste))
print("R2: ", r2_teste)

In [ ]:

# column_transformer = ColumnTransformer(
#     transformers=[
#         ("num", StandardScaler(), numeric_columns)
#     ],
#     remainder="passthrough"
# )
preprocessador = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2))
    ]
)

column_transformer = ColumnTransformer(
    [
        ("num", preprocessador, numeric_columns)
    ]
)

pipeline = Pipeline(
    [("preprocessador", column_transformer),
     ("regressor", Ridge())
    ]
)


In [ ]:

pipeline.fit(X_train, y_train)

train_predict = pipeline.predict(X_train)

mse = mean_squared_error(y_train, train_predict)
r2 = r2_score(y_train, train_predict)

print("\nPrevisões no conjunto de treino: ")
print("Raiz de MSE: ", np.sqrt(mse))
print("R2: ", r2)


test_predict = pipeline.predict(X_test)

mse_teste = mean_squared_error(y_test, test_predict)
r2_teste = r2_score(y_test, test_predict)

print("\nPrevisões no conjunto de teste: ")
print("Raiz de MSE: ", np.sqrt(mse_teste))
print("R2: ", r2_teste)


Previsões no conjunto de treino: 
Raiz de MSE:  4.93356112723694
R2:  0.9652925848964332

Previsões no conjunto de teste: 
Raiz de MSE:  5.288258219728222
R2:  0.9630753847589406


In [ ]:
cvs = cross_val_score(pipeline, X_test, y_test, cv=5, scoring='neg_root_mean_squared_error')
valores = np.abs(cvs)

print("\nCom RMSE: ")
print("Cross Val Score valores: ", valores)
print("Média dos resultados: ", np.mean(valores))

cvs = cross_val_score(pipeline, X_test, y_test, cv=5, scoring='r2')
valores = np.abs(cvs)

print("\nCom R2: ")
print("Cross Val Score valores: ", valores)
print("Média dos resultados: ", np.mean(valores))

In [ ]:
#Agora usando o grid search para definicão de parametros

parametros_opcoes_rigde = {
    'alpha': [0.1, 0.3, 0.5, 0.7, 1.0],
    'fit_intercept': [True, False],
    'copy_X': [True, False],
}

rigde = Ridge()
grid_search = GridSearchCV(rigde, parametros_opcoes_rigde, cv=5, scoring='r2')

X_train_poly = preprocessador.fit_transform(X_train)
grid_search.fit(X_train_poly, y_train)

melhores_parametros = grid_search.best_params_
print("Melhores parametros: ", melhores_parametros)



Melhores parametros:  {'alpha': 0.7, 'copy_X': True, 'fit_intercept': True}


In [ ]:
# Melhores parametros PolynomialFeatures

parametros_opcoes_poly = {
    'poly__degree': [2, 3, 4]
}

grid_search_poly = GridSearchCV(pipeline, parametros_opcoes_poly, cv=5, scoring='r2')


grid_search_poly.fit(X_train, y_train)

melhores_parametros_poly = grid_search_poly.best_params_
print("Melhores parametros: ", melhores_parametros_poly['poly__degree'])

Melhores parametros:  4


In [ ]:

pipeline_completo = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures()),
    ('model', Ridge())
])

parametros_opcoes_poly = {
    'poly__degree': [2, 3, 4]
}

grid_search_poly = GridSearchCV(pipeline_completo, parametros_opcoes_poly, cv=5, scoring='r2')

# Agora sim, fit com X_train (não escalado)
grid_search_poly.fit(X_train, y_train)

melhores_parametros_poly2 = grid_search_poly.best_params_
print("Melhores parâmetros: ", melhores_parametros_poly2)

Melhores parâmetros:  {'poly__degree': 4}


In [ ]:
preprocessador2 = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=melhores_parametros_poly2['poly__degree']))
    ]
)

column_transformer2 = ColumnTransformer(
    [
        ("num", preprocessador2, numeric_columns)
    ]
)

pipeline2 = Pipeline(
    [("preprocessador", column_transformer2),
     ("regressor", Ridge(**melhores_parametros))
    ]
)


pipeline2.fit(X_train, y_train)

train_predict = pipeline2.predict(X_train)

mse = mean_squared_error(y_train, train_predict)
r2 = r2_score(y_train, train_predict)

print("\nPrevisões no conjunto de treino: ")
print("Raiz de MSE: ", np.sqrt(mse))
print("R2: ", r2)


test_predict = pipeline2.predict(X_test)

mse_teste = mean_squared_error(y_test, test_predict)
r2_teste = r2_score(y_test, test_predict)

print("\nPrevisões no conjunto de teste: ")
print("Raiz de MSE: ", np.sqrt(mse_teste))
print("R2: ", r2_teste)

cvs = cross_val_score(pipeline2, X_test, y_test, cv=5, scoring='neg_root_mean_squared_error')
valores = np.abs(cvs)

print("\nCom RMSE: ")
print("Cross Val Score valores: ", valores)
print("Média dos resultados: ", np.mean(valores))

cvs = cross_val_score(pipeline2, X_test, y_test, cv=5, scoring='r2')
valores = np.abs(cvs)

print("\nCom R2: ")
print("Cross Val Score valores: ", valores)
print("Média dos resultados: ", np.mean(valores))